# HN Predictions and Evaluation

Loads the trained AG News classifier and applies it to the
collected Hacker News messages. Produces:
- Predictions and probabilities for all cleaned messages
- Five examples per predicted class
- A 50-message sample for manual accuracy evaluation

## Setup

In [1]:
from pathlib import Path

def find_root(start):
    p = Path(start).resolve()
    for parent in [p, *p.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise FileNotFoundError("Could not find repo root")

ROOT = find_root(__file__ if "__file__" in dir() else Path.cwd())
DATA = ROOT / "data"
MODELS = ROOT / "models"

from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel

spark = (SparkSession.builder
         .appName("hn-predictions")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

model = PipelineModel.load(str(MODELS / "agnews_lr_pipeline"))
print("Model loaded OK")
print(f"Pipeline stages: {[type(s).__name__ for s in model.stages]}")

Model loaded OK
Pipeline stages: ['RegexTokenizer', 'StopWordsRemover', 'CountVectorizerModel', 'IDFModel', 'LogisticRegressionModel']


## Load Raw Messages

In [2]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

hn_schema = StructType([
    StructField("id", StringType()),
    StructField("source", StringType()),
    StructField("type", StringType()),
    StructField("author", StringType()),
    StructField("text", StringType()),
    StructField("timestamp", StringType()),
    StructField("url", StringType()),
    StructField("parent_id", StringType()),
])

raw = spark.read.schema(hn_schema).json(str(DATA / "hn_historical.jsonl"))
print(f"Raw records: {raw.count()}")
raw.groupBy("type").count().show()
raw.select("text").show(5, truncate=80)

Raw records: 1000
+-------+-----+
|   type|count|
+-------+-----+
|  story|  454|
|comment|  546|
+-------+-----+

+--------------------------------------------------------------------------------+
|                                                                            text|
+--------------------------------------------------------------------------------+
|Show HN: Anathem – local memory for coding agents, no LLM and no daemon hi, a...|
|                                 Cherrydock: Design Jobs Handpicked by Designers|
|Ask HN: Did anybody used F# in a production? Curious to apply functional prog...|
|                      Platform engineering needs to become more human...and less|
|                         United Airlines Is Banning AI Bots from Booking Flights|
+--------------------------------------------------------------------------------+
only showing top 5 rows



## Clean Data

In [3]:
from pyspark.sql.functions import col, trim, length, split, size

cleaned = (raw
    .filter(col("text").isNotNull())
    .filter(trim(col("text")) != "")
    .filter(col("text") != "TK")
    .filter(size(split(trim(col("text")), r"\s+")) >= 3)
    .dropDuplicates(["id"])
    .dropDuplicates(["text"]))

print(f"After cleaning: {cleaned.count()} records")
print(f"Dropped: {raw.count() - cleaned.count()} records")
cleaned.groupBy("type").count().show()

After cleaning: 955 records
Dropped: 45 records
+-------+-----+
|   type|count|
+-------+-----+
|  story|  436|
|comment|  519|
+-------+-----+



## Predict and Save

In [4]:
from pyspark.ml.functions import vector_to_array

label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
name_col = F.create_map([F.lit(x) for kv in label_names.items() for x in kv])

preds = model.transform(cleaned)

output = (preds
    .withColumn("prob_array", vector_to_array(col("probability")))
    .withColumn("prob_World",    col("prob_array")[0])
    .withColumn("prob_Sports",   col("prob_array")[1])
    .withColumn("prob_Business", col("prob_array")[2])
    .withColumn("prob_SciTech",  col("prob_array")[3])
    .withColumn("predicted_class", name_col[col("prediction").cast("int")])
    .select("id", "type", "text", "predicted_class", "prediction",
            "prob_World", "prob_Sports", "prob_Business", "prob_SciTech"))

output.write.mode("overwrite").parquet(str(DATA / "hn_predictions.parquet"))
print(f"Predictions saved: {output.count()} records")
output.groupBy("predicted_class").count().orderBy("predicted_class").show()

Predictions saved: 955 records
+---------------+-----+
|predicted_class|count|
+---------------+-----+
|       Business|   58|
|       Sci/Tech|  796|
|         Sports|   76|
|          World|   25|
+---------------+-----+



## Five Examples per Class

In [5]:
from pyspark.sql.window import Window

w = Window.partitionBy("predicted_class").orderBy(F.rand(seed=42))

examples = (output
    .withColumn("rn", F.row_number().over(w))
    .filter(col("rn") <= 5)
    .select("predicted_class", "text", "prob_World", "prob_Sports",
            "prob_Business", "prob_SciTech")
    .orderBy("predicted_class", "rn"))

examples.show(20, truncate=100)
examples.write.mode("overwrite").parquet(str(DATA / "hn_examples.parquet"))

+---------------+----------------------------------------------------------------------------------------------------+---------------------+--------------------+-------------------+-------------------+
|predicted_class|                                                                                                text|           prob_World|         prob_Sports|      prob_Business|       prob_SciTech|
+---------------+----------------------------------------------------------------------------------------------------+---------------------+--------------------+-------------------+-------------------+
|       Business|                                                           Ubuntu package registry Launchpad is down|  0.21102066054795635| 0.20219590117794284| 0.3010440370676599|0.28573940120644076|
|       Business|                                       Muhammad on X: "Claude reverse engineered my 32-year-old Jeep|   0.2584538804825209| 0.24786223917747555|0.26608913233839876|0.227594748

## Manual Label Sample

In [6]:
from pyspark.sql import functions as F

sample = (cleaned
    .orderBy(F.rand(seed=42))
    .limit(50)
    .select("id", "text"))

sample.toPandas().to_csv(str(DATA / "hn_manual_label_sample_done.csv"), index=False)
print("Saved 50 messages to data/hn_manual_label_sample_done.csv")
sample.show(10, truncate=80)

Saved 50 messages to data/hn_manual_label_sample_done.csv
+--------+--------------------------------------------------------------------------------+
|      id|                                                                            text|
+--------+--------------------------------------------------------------------------------+
|50456060|                                                  He earned his bonus that year.|
|50454277|I have a few reasons I'm worried about companies like Anthropic. The first is...|
|50452934|                       Splatt3rs: A Splatoon 3 Automatic Mailbox Message Plotter|
|50455834|In my opinion, a better & more comprehensive look at a similar idea for all s...|
|50453171|                Attention to Mamba: A Recipe for Cross-Architecture Distillation|
|50453621|                             Saw this on X. Pretty cool. Does he get to name it?|
|50456312|The original definition was difference between the best and the worst perform...|
|50453291|            

## Manual Label Accuracy

In [10]:
import pandas as pd
check = pd.read_csv(DATA / "hn_manual_label_sample_done.csv", dtype={"id": str})
print(check.columns.tolist())
print(check.shape)
check.head()

['id', 'text', 'label']
(50, 3)


,id,text,label
0,50456060,He earned his bonus that year.,Business
1,50454277,I have a few reasons I'm worried about compani...,World
2,50452934,Splatt3rs: A Splatoon 3 Automatic Mailbox Mess...,Sci/Tech
3,50455834,"In my opinion, a better & more comprehensive l...",Sci/Tech
4,50453171,Attention to Mamba: A Recipe for Cross-Archite...,Sci/Tech


In [11]:
import pandas as pd

# Load the hand-labeled sample (id as string to match the Spark id column)
labeled = pd.read_csv(DATA / "hn_manual_label_sample_done.csv", dtype={"id": str})
labeled["label"] = labeled["label"].str.strip()

# Pull predicted_class for just the labeled ids from the predictions output
sample_ids = labeled["id"].tolist()
preds_pd = (output
    .filter(col("id").isin(sample_ids))
    .select("id", "predicted_class")
    .toPandas())

joined = labeled.merge(preds_pd, on="id", how="inner")
if len(joined) < len(labeled):
    print(f"WARNING: {len(labeled) - len(joined)} labeled ids had no prediction")

# Overall accuracy
joined["correct"] = joined["label"] == joined["predicted_class"]
total = len(joined)
n_correct = int(joined["correct"].sum())
accuracy = n_correct / total * 100 if total else 0.0

# Correct vs incorrect per (true) class
breakdown = (joined
    .assign(result=joined["correct"].map({True: "correct", False: "incorrect"}))
    .pivot_table(index="label", columns="result", values="id",
                 aggfunc="count", fill_value=0)
    .reindex(columns=["correct", "incorrect"], fill_value=0))
breakdown["total"] = breakdown["correct"] + breakdown["incorrect"]
breakdown["accuracy_%"] = (breakdown["correct"] / breakdown["total"] * 100).round(1)
breakdown.columns.name = None
display(breakdown)

# Save joined table with the overall accuracy attached
joined["overall_accuracy_pct"] = round(accuracy, 1)
joined.to_csv(DATA / "hn_accuracy_results.csv", index=False)
print("Saved data/hn_accuracy_results.csv")

# Summary
AGNEWS_TEST_ACC = 91.2
print("""
=== HN Manual Evaluation Summary ===
Total labeled: {}
Correct:       {}
Accuracy:      {:.1f}%
AG News test:  {}%  (gap: {:+.1f} pts)

Note: The 91.2% AG News test accuracy measures performance on formal news
articles similar to the training data. HN messages are short, informal, and
mostly tech-focused, so a lower accuracy here reflects domain shift rather
than a bug. With only 50 labeled rows each message is worth 2 points, so
this estimate is rough (roughly +/-14 pts at 95% confidence).
""".format(total, n_correct, accuracy, AGNEWS_TEST_ACC, AGNEWS_TEST_ACC - accuracy))

,correct,incorrect,total,accuracy_%
label,,,,
Business,1,7,8,12.5
Sci/Tech,29,2,31,93.5
World,0,11,11,0.0


Saved data/hn_accuracy_results.csv

=== HN Manual Evaluation Summary ===
Total labeled: 50
Correct:       30
Accuracy:      60.0%
AG News test:  91.2%  (gap: +31.2 pts)

Note: The 91.2% AG News test accuracy measures performance on formal news
articles similar to the training data. HN messages are short, informal, and
mostly tech-focused, so a lower accuracy here reflects domain shift rather
than a bug. With only 50 labeled rows each message is worth 2 points, so
this estimate is rough (roughly +/-14 pts at 95% confidence).



In [17]:
# pip install scikit-learn

In [16]:
import sklearn
from sklearn.metrics import classification_report
print(classification_report(joined["label"], joined["predicted_class"], 
                            zero_division=0))


              precision    recall  f1-score   support

    Business       0.33      0.12      0.18         8
    Sci/Tech       0.63      0.94      0.75        31
      Sports       0.00      0.00      0.00         0
       World       0.00      0.00      0.00        11

    accuracy                           0.60        50
   macro avg       0.24      0.27      0.23        50
weighted avg       0.44      0.60      0.50        50

